# Task 1 — Final training and prediction runner

Run this notebook to produce the final Task 1 model and results. It calls the shared Python helpers; the training loop stays in `src/fashion/task1/`.

**Order:** plain CNN refit → label-free holdout prediction → explicit holdout scoring → assignment prediction.

Use the local repository `.venv` kernel and a CUDA GPU. Images are read from this repository. The model, receipts and run log are saved here.

Completed stages are verified and reused. Failed or partial stages stop for inspection. Do not remove their receipts to tune on holdout. Notebook `02_task1_final_eval.ipynb` separately reads and explains these results; shared Notebook 06 remains unchanged.

## 1. Use the local code and output folder

Work in the current MLA2 repository. Once training starts, keep its code version fixed. All outputs stay in this repository.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src/fashion").is_dir() and (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the MLA2 repository.")
required = ["src/fashion/task1/refit.py", "src/fashion/task1/final_evaluation_runner.py",
            "configs/task1/final_evaluation.json"]
if any(not (PROJECT_ROOT / path).is_file() for path in required):
    raise RuntimeError("Final-run code is missing from the local repository. Restore the required files before training.")
os.chdir(PROJECT_ROOT)
os.environ["FASHION_PROJECT_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project and persistent outputs:", PROJECT_ROOT)
print("Code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Project and persistent outputs: C:\Users\Khoa\Documents\MLA2
Code commit: c1e194384c0cdba4d0d2e5f3bdb771e8953b2f86


## 2. Check the local GPU

Select the repository `.venv` kernel with CUDA-enabled PyTorch. This notebook stops if PyTorch cannot use the local GPU. Training uses the GPU; final evaluation currently uses CPU and can take longer.

In [2]:
import json
import pandas as pd
import torch
from IPython.display import display
from fashion.data.dataset import load_splits, load_manifest
from fashion.task1.refit import run_task1_refit
from fashion.task1.final_evaluation_runner import (
    predict_holdout, score_holdout, predict_test, audit_final_evaluation,
)

if not torch.cuda.is_available():
    raise RuntimeError("The local CUDA GPU is unavailable. Select the repository .venv kernel and check that PyTorch has CUDA support before training.")
print("PyTorch:", torch.__version__)
print("Training device:", torch.cuda.get_device_name(0))
contract = json.loads((PROJECT_ROOT / "configs/task1/final_evaluation.json").read_text())
display(contract["refit"])

PyTorch: 2.12.1+cu130
Training device: NVIDIA GeForce RTX 2070 SUPER


{'augmentation_used': False,
 'batch_size': 128,
 'candidate_id': 'task1_cnn_no_aug_unweighted_v1',
 'checkpoint_rule': 'fixed_last_epoch',
 'epochs': 20,
 'grad_clip_norm': 1.0,
 'loss_id': 'cross_entropy_unweighted_v1',
 'max_lr': 0.001,
 'model_family': 'task1_small_cnn_v1',
 'normalization_scope': 'development_only',
 'preprocessing_id': 'task1_rgb_60x80_no_aug_v1',
 'schema_version': 1,
 'scratch': True,
 'seed': 2753,
 'target': 'articleType',
 'validation_used': False,
 'weight_decay': 1e-05}

## 3. Check the local data

Read the canonical split, class map and teacher files from this repository. No new split is made. Stage helpers verify image hashes.

Teacher training and test images, the raw training CSV and the official prediction template must already be present under `data/raw/teacher/`. Restore any missing local files before continuing.

In [3]:
splits = load_splits(PROJECT_ROOT / "data/processed/splits.csv")
official = load_manifest(PROJECT_ROOT / "data/processed/prediction_manifest.csv")
needed = set(splits.loc[splits.partition.isin(["development", "holdout"]), "path"])
needed.update(official["path"])
needed.add("data/raw/teacher/train/styles_train.csv")
needed.add("data/raw/teacher/test/styles_prediction.csv")
missing = {str(path) for path in needed if not (PROJECT_ROOT / str(path)).is_file()}
if missing:
    raise FileNotFoundError(f"{len(missing)} required local data files are missing. Restore them under data/raw/teacher before continuing. Examples: {sorted(missing)[:5]}")
print("Saved split counts:", splits.partition.value_counts().to_dict())
print("Official images:", len(official))

Saved split counts: {'development': 32773, 'holdout': 5778, 'quarantine': 61}
Official images: 5829


## 4. Why save epoch 20?

The selected plain CNN's best validation epochs were **20, 19, 18, 19, 20** across the five development folds. Their median is 19. Epoch 20 mean macro-F1 was about **0.5302**, versus **0.5315** using each fold's best checkpoint: a difference of about 0.0012.

Keep the full tested **20-epoch OneCycle schedule** and save its last epoch. This avoids changing the schedule for a very small observed development difference. All best epochs were near the end, but these runs do not prove that 20 is the best possible budget or that more epochs would improve results.

The budget is fixed before Task 1 holdout scoring. Train from scratch on all **32,773 development images**, using plain RGB input and ordinary unweighted cross-entropy. No validation pass, early stopping or holdout-based model choice is used.

In [4]:
histories = pd.read_csv(PROJECT_ROOT / "results/evidence/task1/cnn_learning_histories.csv")
selected = histories.loc[histories.candidate_id.eq("task1_cnn_no_aug_unweighted_v1")]
summary = []
for fold, frame in selected.groupby("fold"):
    best = frame.loc[frame.macro_f1.idxmax()]
    final = frame.loc[frame.epoch.eq(20)].iloc[0]
    summary.append({"fold": int(fold), "best_epoch": int(best.epoch),
                    "best_macro_f1": best.macro_f1, "epoch20_macro_f1": final.macro_f1})
display(pd.DataFrame(summary))
assert len(summary) == 5 and contract["refit"]["epochs"] == 20

,fold,best_epoch,best_macro_f1,epoch20_macro_f1
0,0,20,0.498052,0.498052
1,1,19,0.519482,0.518614
2,2,18,0.517600,0.514778
3,3,19,0.536994,0.534548
4,4,20,0.585240,0.585240


## 5. Train the final plain CNN

**This cell starts real training.** It saves the model, normalization, training history and a new registry row. A completed run is loaded and checked instead of being trained again. Do not use the holdout to select an epoch.

In [5]:
from dataclasses import asdict

refit = run_task1_refit(project_root=PROJECT_ROOT, mode="run_or_load")
display(asdict(refit))
history = pd.read_csv(PROJECT_ROOT / "results/evidence/task1/development_refit/training_history.csv")
assert history.epoch.tolist() == list(range(1, 21))
assert history.train_samples.eq(32773).all()
display(history)
print("Saved model:", PROJECT_ROOT / "models/task1_article_type.pt")

{'source': 'trained',
 'run_id': 'task1-cnn-task1_cnn_no_aug_unweighted_v1-final-refit-fall-s2753-25828f9946dd',
 'manifest_path': 'C:\\Users\\Khoa\\Documents\\MLA2\\models\\task1_article_type.manifest.json',
 'manifest_sha256': '6000ce48f8d31e8c1ef43518fb4c42096fa124d84d0f93219d61398d9afb8395',
 'bundle_path': 'C:\\Users\\Khoa\\Documents\\MLA2\\models\\task1_article_type.pt',
 'bundle_sha256': 'c3d219980590a76aab7c58ad4d564a525d243056183d7857de2ac15ff38000ad',
 'final_epoch': 20,
 'development_rows': 32773}

,epoch,train_loss,train_samples,learning_rate
0,1,3.436021,32773,1.043894e-04
1,2,1.943272,32773,2.802825e-04
2,3,1.267751,32773,5.204893e-04
3,4,0.978827,32773,7.605648e-04
4,5,0.805609,32773,9.360993e-04
5,6,0.695850,32773,9.999998e-04
6,7,0.585571,32773,9.873667e-04
7,8,0.505024,32773,9.502950e-04
8,9,0.434581,32773,8.906438e-04
9,10,0.374709,32773,8.114042e-04


Saved model: C:\Users\Khoa\Documents\MLA2\models\task1_article_type.pt


## 6. Save internal holdout predictions before labels

This stage saves all 124 probabilities for each of the **5,778 holdout images**, plus the fixed photo tests. It excludes quarantine. The model, input files and code are tied to hashes before scoring. It does not read raw holdout labels. CPU timing is measured here too.

In [6]:
EVIDENCE_DIR = PROJECT_ROOT / "results/evidence/task1/final_evaluation"
if not (EVIDENCE_DIR / "prediction_receipt.json").exists():
    predict_holdout(PROJECT_ROOT)
else:
    print("Saved prediction stage found. The scoring/audit stage below verifies its hashes.")
print("Prediction receipt:", EVIDENCE_DIR / "prediction_receipt.json")

Prediction receipt: C:\Users\Khoa\Documents\MLA2\results\evidence\task1\final_evaluation\prediction_receipt.json


## 7. Explicitly unlock and score the saved holdout predictions

**This cell opens protected labels.** Run it only after the final model and predictions above are fixed. Other targets have already used this shared holdout; do not call it untouched for the whole group.

It scores the same frozen model, with no training or threshold selection. Save the findings even if performance is weaker than expected. Repeating a completed stage audits the saved evidence.

In [7]:
if not (EVIDENCE_DIR / "evaluation_manifest.json").exists():
    score_holdout(PROJECT_ROOT, evaluation_unlocked=True)
evidence = audit_final_evaluation(PROJECT_ROOT)
display(evidence["tables"]["metrics"])
display(evidence["tables"]["uncertainty"])
display(evidence["tables"]["confusion_pairs"].head(15))
print("Read the full analysis in notebooks/02_task1_final_eval.ipynb.")

,rows,macro_f1,weighted_f1,top1_accuracy,top5_accuracy
0,5778,0.575221,0.846932,0.849256,0.984251


,metric,estimate,lower_95,upper_95,families,replicates
0,macro_f1,0.575221,0.479853,0.555271,4110,10000
1,weighted_f1,0.846932,0.835903,0.857899,4110,10000
2,top1_accuracy,0.849256,0.838635,0.859802,4110,10000
3,top5_accuracy,0.984251,0.980409,0.987729,4110,10000


,articleType,predicted_label,errors,true_class_support,share_of_true_class
0,Tshirts,Tops,59,1020,0.057843
1,Sports Shoes,Casual Shoes,42,300,0.140000
2,Casual Shoes,Sports Shoes,41,404,0.101485
3,Tops,Tshirts,39,243,0.160494
4,Flats,Heels,23,62,0.370968
5,Heels,Flats,21,158,0.132911
6,Shirts,Tshirts,17,464,0.036638
7,Sandals,Flip Flops,17,130,0.130769
8,Formal Shoes,Casual Shoes,15,91,0.164835
9,Tops,Shirts,14,243,0.057613


Read the full analysis in notebooks/02_task1_final_eval.ipynb.


## 8. Predict the assignment test set

Use the same checkpoint for **5,829 official images**. This stage writes `results/article_type_test_predictions.csv` with `id,articleType` in template order. The test set has no supplied labels, so this is prediction, not an accuracy score.

The group still needs to validate and merge the other owners' columns. Keep the original `styles_prediction.csv` template unchanged.

In [8]:
if not (EVIDENCE_DIR / "test_prediction_receipt.json").exists():
    predict_test(PROJECT_ROOT)
evidence = audit_final_evaluation(PROJECT_ROOT)
display(evidence["test_prediction_receipt"])
predictions = pd.read_csv(PROJECT_ROOT / "results/article_type_test_predictions.csv", keep_default_na=False)
assert len(predictions) == 5829
assert list(predictions.columns) == ["id", "articleType"]
display(predictions.head())

{'combined_submission_status': 'pending validated outputs from the other owners; source template unchanged',
 'completed_at': '2026-09-09T03:22:53.969112+00:00',
 'evaluation_manifest': {'path': 'results/evidence/task1/final_evaluation/evaluation_manifest.json',
  'sha256': '50c5fef00f14928bbf4a4513fb953dc8a83248c37cb0dcb0dd6b757f06a2f50d'},
 'image_ids_sha256': 'ac9daefef24fdcbf551794fa0a81829dd4bddb1d967fe8e9bda526c2eb147595',
 'images': [{'id': 52003,
   'path': 'data/raw/teacher/test/images_test/52003.jpg',
   'sha256': '1db9db8531ac96f30563eb50768a1ff895c3c0137ea3704fc4c2ecdac0ba0d5c'},
  {'id': 52007,
   'path': 'data/raw/teacher/test/images_test/52007.jpg',
   'sha256': '34a65eb1a937f9ca424e702c73fec7300b272e00aa61121a77061c62a308f75d'},
  {'id': 52017,
   'path': 'data/raw/teacher/test/images_test/52017.jpg',
   'sha256': '426b2a6fb15bd4adbbf8d4f173b153667044ea5cb845546837434d5c0c4ec467'},
  {'id': 52021,
   'path': 'data/raw/teacher/test/images_test/52021.jpg',
   'sha256': '9

,id,articleType
0,52003,Casual Shoes
1,52007,Nightdress
2,52017,Bra
3,52021,Briefs
4,52023,Tshirts


## 9. Keep and hand off the results

Results are saved in this local repository. Keep `models/task1_article_type.*`, `results/evidence/task1/`, `results/figures/task1/`, `results/article_type_test_predictions.csv` and `results/runs.csv`. Keep the exact code and config too; replay checks their hashes. The training helper appends the final run to the shared local registry.

Then run `02_task1_final_eval.ipynb` to read the full report. Final group CSV merge and app demonstration are separate steps.